In [2]:
# Standard libraries
import os
import json
import bisect
import numpy as np
from typing import Optional

# PyTorch core
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, Sampler

Import training data

The FastVariableShardDataset class is a PyTorch Dataset that efficiently loads samples from a sharded and preprocessed dataset stored on disk. It’s designed to support large-scale training data split across many .pt files — without loading everything into memory.

In [3]:
class FastVariableShardDataset(Dataset):
    """
    Load sharded (inputs, targets) pairs efficiently.
    
    Each shard is a .pt file containing:
        {
            "inputs":  Tensor of shape (B, N, F),
            "targets": Tensor of shape (B, H, W)
        }

    - Inputs are features per convective core.
    - Targets are preprocessed binary maps (already resized if needed).
    - Only one shard is cached in memory at a time for efficiency.
    """

    def __init__(self, 
                 shards_dir: str, 
                 sizes_file: str,
                 output_size: Optional[tuple[int, int]] = (512, 512)):
        
        super().__init__()
        self.shard_dir = shards_dir
        self.output_size = output_size  # Not used in this version (preprocessed only)

        # Gather all shard files in the directory (must end with _proc.pt)
        self.shard_files = sorted(
            f for f in os.listdir(shards_dir) if f.endswith("_proc.pt")
        )
        if not self.shard_files:
            raise FileNotFoundError(f"No *_proc.pt files found in {shards_dir}")

        # Load sample counts per shard from sizes JSON
        with open(sizes_file) as f:
            sizes_dict = json.load(f)

        # Store shard sizes and offsets (for global-to-local index lookup)
        self.shard_sizes = [sizes_dict[f] for f in self.shard_files]
        self.shard_offsets = [0]
        for sz in self.shard_sizes[:-1]:
            self.shard_offsets.append(self.shard_offsets[-1] + sz)
        self.total_samples = sum(self.shard_sizes)

        # One-shard RAM cache to avoid repeatedly loading from disk
        self._cache_path: Optional[str] = None
        self._cache_data: Optional[dict] = None

    def __len__(self) -> int:
        """Return total number of samples across all shards."""
        return self.total_samples

    def _load_shard(self, fname: str) -> dict:
        """
        Load a single shard file and cache it in memory.
        If the requested shard is already cached, reuse it.
        """
        if fname != self._cache_path:
            path = os.path.join(self.shard_dir, fname)
            data = torch.load(path, map_location="cpu")
            self._cache_data = data
            self._cache_path = fname
        return self._cache_data

    def __getitem__(self, idx: int):
        """
        Fetch a single sample by global index.
        Automatically finds the correct shard and index within it.
        Returns:
            x: Tensor of shape (N, F)
            y: Tensor of shape (H, W)
        """
        if idx < 0 or idx >= self.total_samples:
            raise IndexError(f"Index {idx} out of bounds")

        # Binary search to find the correct shard for this global index
        shard_idx = bisect.bisect_right(self.shard_offsets, idx) - 1
        local_idx = idx - self.shard_offsets[shard_idx]

        # Load the shard (from disk or cache)
        shard_data = self._load_shard(self.shard_files[shard_idx])

        # Extract the sample
        x = shard_data["inputs"][local_idx].float()    # shape: (T, F)
        y = shard_data["targets"][local_idx].float()   # shape: (H, W)

        return x, y

In [4]:
LEAD_TIME = 1

# === Validation ===
VAL_PARTITION = "val"
VAL_SHARDS_DIR = f"/work/scratch-nopw2/mrakotomanga/eps/pancast-shard-processed/t{LEAD_TIME}/{VAL_PARTITION}"
VAL_SIZES_FILE = f"/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/model/africa/shard-analysis/t{LEAD_TIME}/{VAL_PARTITION}/{VAL_PARTITION}-t{LEAD_TIME}-shards-proc-sizes.json"

val_ds = FastVariableShardDataset(
    shards_dir=VAL_SHARDS_DIR,
    sizes_file=VAL_SIZES_FILE,
    output_size=None
)

val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=16, shuffle=False, num_workers=8, pin_memory=True
)

# === Test ===
TEST_PARTITION = "test"
TEST_SHARDS_DIR = f"/work/scratch-nopw2/mrakotomanga/eps/pancast-shard-processed/t{LEAD_TIME}/{TEST_PARTITION}"
TEST_SIZES_FILE = f"/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/model/africa/shard-analysis/t{LEAD_TIME}/{TEST_PARTITION}/{TEST_PARTITION}-t{LEAD_TIME}-shards-proc-sizes.json"

test_ds = FastVariableShardDataset(
    shards_dir=TEST_SHARDS_DIR,
    sizes_file=TEST_SIZES_FILE,
    output_size=None
)

test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=16, shuffle=False, num_workers=8, pin_memory=True
)

In [5]:
def sanity_check_loader(loader, n_batches_to_check=2, binary_targets=True):
    """
    Perform a basic sanity check on a DataLoader.

    Args:
        loader: torch.utils.data.DataLoader
        n_batches_to_check: number of batches to check (default: 2)
        binary_targets: if True, checks that y is binary (0 or 1)
    """
    for batch_idx, (x, y) in enumerate(loader):
        print(f"[Batch {batch_idx}]")
        print(f"   x shape: {x.shape}")  # e.g., (B, N, F)
        print(f"   y shape: {y.shape}")  # e.g., (B, H, W)

        if torch.isnan(x).any():
            print(f"NaNs in input")
        if torch.isnan(y).any():
            print(f"NaNs in target")
        if binary_targets and not ((y == 0) | (y == 1)).all():
            print(f"Targets not binary — min={y.min().item()}, max={y.max().item()}")

        if batch_idx + 1 >= n_batches_to_check:
            break

    print("✅ Sanity check complete.")

In [6]:
sanity_check_loader(val_loader, n_batches_to_check=20)

[Batch 0]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 1]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 2]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 3]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 4]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 5]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 6]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 7]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 8]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 9]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 10]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 11]
   x shape: torch.Si

In [7]:
import random
from torch.utils.data import Sampler

class ShardedSampler(Sampler):
    """
    Custom sampler for sharded datasets.

    Instead of shuffling the entire dataset globally (which would require random disk access),
    this sampler:
      - Shuffles samples **within** each shard
      - Preserves **sequential shard-level access**
      - Optionally applies a per-epoch seed for reproducible shuffling

    This greatly improves performance for datasets stored in separate `.pt` shard files.

    Args:
        shard_sizes: List[int] — number of samples in each shard
        batch_size:  Optional — not used directly, but can inform batching externally
        shuffle:     Whether to shuffle within each shard
        seed:        Optional random seed to control shuffling behaviour (e.g. per epoch)
    """

    def __init__(self, shard_sizes, batch_size=16, shuffle=True, seed=None):
        self.shard_sizes = shard_sizes
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed

        # Compute cumulative offsets to map global indices to shards
        self.shard_offsets = [0]
        for sz in shard_sizes[:-1]:
            self.shard_offsets.append(self.shard_offsets[-1] + sz)

    def __iter__(self):
        rng = np.random.default_rng(self.seed) if self.shuffle else None
        indices = []

        for shard_idx, size in enumerate(self.shard_sizes):
            offset = self.shard_offsets[shard_idx]
            local_indices = np.arange(offset, offset + size)

            if self.shuffle:
                rng.shuffle(local_indices)

            indices.extend(local_indices.tolist())

        return iter(indices)


    def __len__(self):
        """Total number of samples across all shards."""
        return sum(self.shard_sizes)


In [8]:
TRAIN_PARTITION = "train"
TRAIN_SHARDS_DIR = f"/work/scratch-nopw2/mrakotomanga/eps/pancast-shard-processed/t{LEAD_TIME}/{TRAIN_PARTITION}"
TRAIN_SIZES_FILE = f"/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/model/africa/shard-analysis/t{LEAD_TIME}/{TRAIN_PARTITION}/{TRAIN_PARTITION}-t{LEAD_TIME}-shards-proc-sizes.json"

train_ds = FastVariableShardDataset(
    shards_dir=TRAIN_SHARDS_DIR,
    sizes_file=TRAIN_SIZES_FILE,
    output_size=None
)

train_sampler = ShardedSampler(
    shard_sizes=train_ds.shard_sizes,
    batch_size=16,
    shuffle=False,
    seed=None  # set per epoch if needed
)

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=16, sampler=train_sampler, num_workers=8, pin_memory=True
)

In [9]:
sanity_check_loader(train_loader, n_batches_to_check=4)

[Batch 0]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 1]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 2]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
[Batch 3]
   x shape: torch.Size([16, 140, 9])
   y shape: torch.Size([16, 1024, 1024])
✅ Sanity check complete.
